# Spatial Statistics: Dependence and Validation

Nearby observations are often more similar than distant observations. This notebook shows why random cross-validation can be too optimistic and why spatial blocking is useful.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, KFold, cross_val_score

rng = np.random.default_rng(5)
n = 500
coords = rng.uniform(0, 1, size=(n, 2))
x, y = coords[:, 0], coords[:, 1]
target = np.sin(3*np.pi*x) + np.cos(3*np.pi*y) + 0.5*np.sin(5*np.pi*(x+y)) + rng.normal(scale=0.20, size=n)
model = RandomForestRegressor(n_estimators=150, min_samples_leaf=3, random_state=11, n_jobs=1)
random_cv = KFold(5, shuffle=True, random_state=13)
random_rmse = np.sqrt(-cross_val_score(model, coords, target, cv=random_cv, scoring='neg_mean_squared_error'))
groups = np.minimum((x*5).astype(int), 4)
block_cv = GroupKFold(5)
block_rmse = np.sqrt(-cross_val_score(model, coords, target, cv=block_cv, groups=groups, scoring='neg_mean_squared_error'))
pd.DataFrame({'scheme':['random CV','spatial block CV'], 'mean_rmse':[random_rmse.mean(),block_rmse.mean()], 'sd_rmse':[random_rmse.std(ddof=1),block_rmse.std(ddof=1)]})


Spatial blocking better reflects prediction at locations separated from the training data. Validation design should match the dependence structure that will exist at deployment time.
